In [ ]:
import numpy as np
import filters_exp as exp
from matplotlib import pyplot as plt
from numba import njit
from numba_progress import ProgressBar

from filters import (
    # parameter dtypes
    NLMS_params, sKF_params, sKF_L_params, skf_int_params, skf_L_int_params,
    # signal / environment helpers
    autocorr_matrix_calc, autocorr_matrix_estimate, AR_settling_time, std_behavior,
    # algorithms
    NLMS_algorithm, sKF_algorithm, sKF_L_algorithm, sKF_L_exact_algorithm,
    sKF_integral_algorithm, sKF_L_integral_algorithm,
    # monte carlo driver
    MC_Simulations_Modular_Variance,
)

In [ ]:
NR = 200
N = int(100e3)
L = 64
ho = np.sinc(np.linspace(0,1,L))
ho = ho/np.linalg.norm(ho)
h0 = np.zeros(L)
var_x = 1
var_v = 1e-3
#AR = np.array([1.0, 0.0])
#AR = np.array([1.0, -0.6, 0.85])
AR = np.array([1.0, -0.9, 0.95, -0.8, 0.8])

mu = 0.1
delta = 1e-3

NLMS_Parameters = np.void(("NLMS", mu, delta), dtype=NLMS_params)
env_parameters = exp.std_env_parameters(ho=ho, AR=AR, var_v=var_v, var_x=var_x)

Algorithms = [NLMS_algorithm]
Alg_Parameters = [NLMS_Parameters]

with ProgressBar(total=NR) as PBar:
    MC_measures = exp.MC_Simulations(N, 
                                    NR, 
                                    env_parameters,
                                    exp.std_behavior,
                                    Algorithms,
                                    Alg_Parameters,
                                    h0,
                                    PBar = PBar)

In [ ]:
for params in Alg_Parameters:
    label = params["label"]
    plt.plot(10*np.log10(MC_measures[label]['Jex']))
plt.ylabel("EMSE (dB)")
plt.xlabel("Iterations")
plt.show()